## Exercise 3: Misspecification testing - solutions

This is the walk-through from the exercise class.

**Three things are deliberately left blank**, because they are the parts you have to
hand in yourself:

* the companion matrix and the stability check (**homework question 6**),
* the $Q_{LM}$ statistic itself (**homework question 4**),
* the Edgerton-Shukur block $m, q, s, N$ and $\mathrm{FLM}_h$ (**homework question 4**).

Everything else is filled in: the estimation calls, the auxiliary regression, the
provided functions, the data handling and all the discussion. Every result you should
get is written out in the markdown, so you can check yourself as you fill in the gaps.

Because of the gaps the notebook does not run top to bottom as it stands. That is
intentional.

---

Consider the following VAR(p)-process:
\begin{equation*}
y_{t} = \nu + \rho_{1}y_{t-1} + \cdots + \rho_{p}y_{t-p} + u_{t}
\end{equation*}
The Breusch-Godfrey idea is to model the residuals as
\begin{equation*}
u_{t} = D_{1}u_{t-1} + \cdots + D_{h}u_{t-h} + e_{t}
\end{equation*}
and test $H_0: D_1 = \cdots = D_h = 0$. Because $\hat{u}_t$ is correlated with the
regressors, the auxiliary regression has to keep the original regressors too:
\begin{equation*}
\hat{u}_{t} = \nu + \rho_{1}y_{t-1} + \cdots + \rho_{p}y_{t-p} + D_{1}\hat{u}_{t-1} + \cdots + D_{h}\hat{u}_{t-h} + e_{t}
\end{equation*}
and the statistic is
\begin{equation*}
Q_{LM} = T \left(K - \text{tr}(\widetilde{\Sigma}_{u}^{-1} \widetilde{\Sigma}_{e})\right) \;\overset{a}{\sim}\; \chi^{2}(hK^{2})
\end{equation*}

Throughout, $T$ is the **effective** sample `y.shape[0] - p`.

In [ ]:
%reset -f

import numpy as np
import pandas as pd
from scipy.stats import chi2
from scipy.stats import f
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.diagnostic import het_arch
from statsmodels.stats.stattools import jarque_bera

np.random.seed(1)

# Parameters
N = 100  # Number of observations
rho1 = np.array([[0.8, 0],  # Autoregressive coefficient for lag 1
                 [0, 0.8]])

rho2 = np.array([[0.15, 0],  # Autoregressive coefficient for lag 2
                 [0, 0.15]])

alpha = np.array([0, 0])  # Mean of the VAR process

sigma = np.array([[1, 0],  # Covariance matrix of white noise
                  [0, 1]])

# Make the y-matrix ready to insert values
y = np.zeros((2, N))

# Generate white noise
epsilon = np.random.multivariate_normal([0, 0], sigma, N).T

# Generate the VAR(2) process with time trend
for t in range(2, N):
    y[:, t] = alpha + rho1 @ y[:, t-1] + rho2 @ y[:, t-2] + epsilon[:, t] + 0.1*t

y = y.T

1.-2. Import `VARlsExog` and estimate a VAR(1) with no constant, no trend and no exogenous variables.

In [ ]:
from s_master import VARlsExog

p = 1
con = 0
tr = 0
exog = 0

Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

print(f"{Beta.T}")

print(f"\n The {Beta.T[0,0]} is the effect of variable 1 at t-1 on variable 1 at time t.")
print(f"\n The {Beta.T[0,1]} is the effect of variable 2 at t-1 on variable 1 at time t.")

You should get

    [[0.9893183  0.03109394]
     [0.05101393 0.96630013]]

The columns of `Beta.T` follow the order in which `VARlsExog` builds the regressor
matrix: first all the lags, then the constant, then the trend, then the exogenous
variables. That ordering is what makes the companion matrix below easy to slice out.

3. Companion matrix and stability.

**This is homework question 6, so it is left blank.**

Two things that are easy to get wrong:

* `Beta.T` also contains the deterministic terms, so take only the first $Kp$ columns.
* Sort the moduli in **descending** order, otherwise `e[0]` is the smallest eigenvalue
  and the stability check below silently returns the wrong verdict. Note that this is
  also Bergman's convention: on p. 10 of the lecture 2-5 handout he writes
  $|\lambda_{Kp}| \geq \cdots \geq |\lambda_1|$ and calls the model stable iff the
  largest one is below 1.

In [ ]:
def comp(Beta, p):
    # FILL IN - homework question 6
    return A

A = comp(Beta, p)


def stabVAR(A):
    # FILL IN - homework question 6
    return e

e = stabVAR(A)

print(f"The sorted eigenvalues are: {e}")

if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

The sorted eigenvalues should be: [1.0192662141023392, 0.936352213278791]

Not stable - which is exactly what we should expect. The data were generated with a
deterministic trend and we gave the model nothing to absorb it with, so the estimated
autoregressive root is pushed above one to fake the trend.

That is worth remembering: an apparent unit root can be a *specification* failure
rather than a genuine one.

4. Reestimate with a linear trend (still no constant) and check stability again.

In [ ]:
p = 1
con = 0
tr = 1
exog = 0

Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

print(f"{Beta.T}")
print("\nThe third column is the coefficient on the linear trend in each equation.")

A = comp(Beta, p)
e = stabVAR(A)
print(f"\nThe sorted eigenvalues are: {e}")
if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

Now

    [[0.95561392 0.00831636 0.08062992]
     [0.01062805 0.93900716 0.09661386]]

and the eigenvalues are [0.95985378, 0.9347673], so both moduli are below one and the
model is stable.

Note that `comp` is unaffected by the trend: we slice out the first $Kp = 2$ columns of
`Beta.T` and leave the trend coefficient behind.

5. The LM test, step by step. **This is homework question 4.**

(a) Lag the residuals.

In [ ]:
from s_master import lagmatrix

h = 1

u_lags = lagmatrix(residuals_u, h)

print(u_lags.shape, residuals_u.shape)

Both are (99, 2). `lagmatrix` does not shorten the sample, it fills the first rows
with zeros - which is exactly what we want, because `VARlsExog` expects `exog` to have
the same number of rows as the lagged regressor block.

(b) The auxiliary regression: same $p$ lags of $y$, plus the $h$ lags of the residuals
as exogenous variables.

Pass `p`, not `h`. With $p = h = 1$ you cannot tell the difference, but in question 11 we
use $p = 3$ and $h = 4$ and then passing `h` would not even run - the row counts no longer
line up.

In [ ]:
Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = VARlsExog(y, p, con, tr, u_lags)

(c) The test statistic:

\begin{equation*}
    Q_{LM} = T \left(K - \text{tr}(\widetilde{\Sigma}_{u}^{-1} \widetilde{\Sigma}_{e})\right)
\end{equation*}

**Homework question 4, so left blank.**

Note the effective sample: `y.shape[0]` is 100, but only $100 - p$ rows are used. If you
use 100 you get 23.52 instead of 23.29 here, and 6.30 instead of 6.17 further down.

Useful numpy pieces: `np.linalg.inv`, `np.trace`, and `@` for matrix multiplication.

In [ ]:
t, K = y.shape

LML = # FILL IN - homework question 4

(d) The asymptotic $\chi^2(hK^2)$ p-value.

In [ ]:
LMLpval = 1 - chi2.cdf(LML, h*K**2)

print(round(LML, 2), round(LMLpval, 4))

[LML, LMLpval] = [23.29, 0.0001], so we reject strongly: the VAR(1) does suffer from
residual autocorrelation. That is no surprise, the true process is a VAR(2).

Now extend to p = 2 and h = 2.

In [ ]:
p = 2
con = 0
tr = 1
exog = 0
h = 2

Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

u_lags = lagmatrix(residuals_u, h)
Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = VARlsExog(y, p, con, tr, u_lags)

t, K = y.shape
LML = # FILL IN - same formula as above
LMLpval = 1 - chi2.cdf(LML, h*K**2)

print(round(LML, 2), round(LMLpval, 4))

[LML, LMLpval] = [6.17, 0.6279]. We no longer reject. The VAR(2) has captured the
dynamics, which it should, because we generated the data from a VAR(2).

6. Collect everything into one function.

The wiring is given; only the statistic itself is left for you.

Note the extra return value `residuals_u` - we reuse those residuals for the ARCH and
normality tests further down, so it saves re-estimating.

In [ ]:
def VARLMtest(y, p, con, tr, exog, h):

    t, K = y.shape
    Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

    u_lags = lagmatrix(residuals_u, h)
    Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = VARlsExog(y, p, con, tr, u_lags)

    T = t - p
    LML = # FILL IN - homework question 4
    LMLpval = 1 - chi2.cdf(LML, h*K**2)

    return LML, LMLpval, residuals_u

LML, LMLpval, _ = VARLMtest(y, p, con, tr, exog, h)

print(round(LML, 2), round(LMLpval, 4))

7. Add the Edgerton and Shukur (1999) small sample correction:

\begin{equation*}
\text{FLMh}=\left(\left(\det(\Sigma_{u})\det(\Sigma_{e})^{-1}\right)^{\frac{1}{s}}-1\right)\cdot\left(\frac{Ns-q}{Km}\right)
\end{equation*}

where:

\begin{equation*}
\begin{aligned}
m & =Kh\\
q & =\frac{1}{2}Km-1\\
s & =\left(\frac{K^{2}\cdot m^{2}-4}{K^{2}+m^{2}-5}\right)^{\frac{1}{2}}\\
N & =T-Kp-m-\frac{1}{2}(K-m+1)
\end{aligned}
\end{equation*}

It is $F(Km,\; Ns-q)$ distributed, so the p-value comes from `f.cdf`, not `chi2.cdf`.

**Homework question 4, so left blank.**

Use the same effective $T = t - p$ here as in $Q_{LM}$, otherwise the two statistics are
not computed on the same sample and the numbers below will not match. `np.linalg.det`
and `np.sqrt` are what you need.

In [ ]:
p = 2
con = 0
tr = 1
exog = 0
h = 2

def VARLMtest(y, p, con, tr, exog, h):

    t, K = y.shape
    Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

    u_lags = lagmatrix(residuals_u, h)
    Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = VARlsExog(y, p, con, tr, u_lags)

    T = t - p
    LML = # FILL IN
    LMLpval = 1 - chi2.cdf(LML, h*K**2)

    m = # FILL IN
    q = # FILL IN
    s = # FILL IN
    N = # FILL IN

    FLMh = # FILL IN
    FLMh_pval = # FILL IN


    Results = [[LML, FLMh], [LMLpval, FLMh_pval], [h, h]]

    lm_table = pd.DataFrame({
        'Measure': pd.Categorical(['Test statistic', 'p-value', 'Lag order']),
        'Breusch_Godfrey': [row[0] for row in Results],
        'Edgerton_Shukur': [row[1] for row in Results]
    })

    return Results, lm_table, residuals_u

Results, lm_table, _ = VARLMtest(y, p, con, tr, exog, h)

lm_table

With p = 2 and h = 2 you should get:

| Measure         | Breusch-Godfrey | Edgerton-Shukur |
|-----------------|-----------------|-----------------|
| Test statistic  | 6.17            | 0.73            |
| p-value         | 0.63            | 0.66            |
| Lag order       | 2               | 2               |

Here the two agree. On the real data in question 11 they will not.

8. Univariate autocorrelation, as a cross-check. This is the "check these results with
built-in functions" part of homework question 5.

In [ ]:
p = 2
con = 0
tr = 1
exog = 0

t, K = y.shape

Beta, CovBeta, tratioBeta, residuals, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

lags = [1, 2, 3, 4, 5, 6]

p_values_array = np.zeros((len(lags), K))

for i in range(K):
    df = acorr_ljungbox(residuals[:, i], lags=lags, return_df=True)
    p_values_array[:, i] = df['lb_pvalue'].values

p_values_df = pd.DataFrame(p_values_array, index=[f"P-Value Lag {lag}" for lag in lags],
                           columns=["Variable 1", "Variable 2"])

p_values_df

All p-values come out between 0.45 and 0.99, so comfortably above 0.05: the univariate
tests agree with the multivariate one.

Note what Ljung-Box does *not* do: it looks at each equation separately, so it cannot see
cross-equation autocorrelation. That is the whole reason we code the multivariate test.

9. Multivariate ARCH, using `march()` from s_master.

Degrees of freedom are $qK^2(K+1)^2/4$, which here is $2 \cdot 4 \cdot 9/4 = 18$.

In [ ]:
from s_master import march

lag = 2

test, march_table = march(residuals, lag, K)

march_table

Test statistic 22.24 on 18 degrees of freedom, p-value 0.22. No ARCH, which is right -
we drew the errors from a fixed-variance normal.

... and univariate ARCH.

In [ ]:
hypothesis_results = np.zeros(K)
p_values_results = np.zeros(K)
stat_results = np.zeros(K)
crit_value_results = np.zeros(K)

for i in range(K):
    stat, pValue, fval, _ = het_arch(residuals[:, i], nlags=lag)
    hypothesis_results[i] = 1 if pValue < 0.05 else 0
    p_values_results[i] = pValue
    stat_results[i] = stat
    crit_value_results[i] = chi2.ppf(1 - 0.05, lag)

row_names = ['Variable 1', 'Variable 2']

het_table = pd.DataFrame({
    'Hypothesis': hypothesis_results,
    'P-Value': p_values_results,
    'Test statistic': stat_results,
    'CriticalValue': crit_value_results
}, index=row_names)

het_table

p-values 0.55 and 0.30. Nothing rejected, consistent with the multivariate test.

10. Multivariate normality.

As per Kilian and Lütkepohl (2017, p 66), standardize the residuals
$\hat{u}_{t}^{s} = P^{-1} \hat{u}_{t}$ where $\tilde{\Sigma}_{u} = P P^{\prime}$, and test

\begin{equation*}
\lambda_{3} = T\,\frac{\hat{b}_{3}^{\prime} \hat{b}_{3}}{6} \overset{d}{\rightarrow} \chi^{2}(K),
\qquad
\lambda_{4} = T\,\frac{(\hat{b}_{4} - 3_{K})^{\prime} (\hat{b}_{4} - 3_{K})}{24} \overset{d}{\rightarrow} \chi^{2}(K),
\qquad
\lambda = \lambda_{3} + \lambda_{4} \sim \chi^{2}(2K)
\end{equation*}

where $\hat{b}_{kj} = \frac{1}{T} \sum_{t=1}^{T} (\hat{u}_{kt}^{s})^{j}$. This is just
Jarque-Bera one equation at a time, summed, which is why the joint degrees of freedom
are $2K$.

`multnorm` reports two columns because Doornik-Hansen and Lütkepohl whiten the residuals
differently: the first uses the symmetric square root of $\widetilde{\Sigma}_u$, the second
the Cholesky decomposition. Same null hypothesis, two numbers. The Cholesky version depends
on the ordering of the variables; the square-root version does not. That is the reason to
prefer Doornik-Hansen.

In [ ]:
from s_master import multnorm

norm, multnorm_table = multnorm(residuals)

multnorm_table

Joint statistics 1.279 and 1.175 on 4 degrees of freedom, p-values 0.87 and 0.88.
Nothing rejected.

... and univariate Jarque-Bera.

Careful with this loop: the obvious copy-paste error is to keep testing `pValue` from the
ARCH loop above instead of `JBpv`.

In [ ]:
hypothesis_results = np.zeros(K)
p_values_results = np.zeros(K)
stat_results = np.zeros(K)
crit_value_results = np.zeros(K)

for i in range(K):
    (JB, JBpv, skew, kurtosis) = jarque_bera(residuals[:, i])
    hypothesis_results[i] = 1 if JBpv < 0.05 else 0
    p_values_results[i] = JBpv
    stat_results[i] = JB
    crit_value_results[i] = chi2.ppf(1 - 0.05, 2)

row_names = ['Variable 1', 'Variable 2']

jb_table = pd.DataFrame({
    'Hypothesis': hypothesis_results,
    'P-Value': p_values_results,
    'Test statistic': stat_results,
    'CriticalValue': crit_value_results
}, index=row_names)

jb_table

p-values 0.63 and 0.62. The simulated residuals are Gaussian by construction, so
nothing is rejected. Good - that tells us the code is not broken before we point it at
real data.

Note 1: The critical value only approaches $\chi^2(2)$ in large samples, hence the above
Critical Value is not correct in small samples.

Note 2: Asymptotic properties of the univariate tests can be found on Absalon.

11. Now the real thing: Bjørnland (2009), Swedish data, the homework specification.

* $x_t = [\;rfor,\; lgdp,\; \pi,\; dlrexc,\; rdom\;]$, $K = 5$
* $p = 3$ (from the information criteria and the top-down sequence in Exercise 2)
* constant and linear trend
* three exogenous dummies: du93Q1, du92Q3, du95Q4
* 1983:1 - 2004:4, $T = 88$

A copy of `result.csv` is in this folder if you do not have your own from Exercise 1.

In [ ]:
y_df = pd.read_csv('result.csv', index_col=0)

print(y_df.columns.tolist())

y = y_df[['rfor', 'lgdp', 'pi', 'dlrexc', 'rdom']].to_numpy()
dummies_full = y_df[['du93Q1', 'du92Q3', 'du95Q4']].to_numpy()

T, K = y.shape
print(f"T = {T}, K = {K}")

Two adjustments are needed before `VARLMtest` works on this specification.

First, `VARlsExog` stacks `exog` next to the lagged regressors, which only have $T-p$ rows,
so the dummy matrix has to lose its first $p$ rows.

Second - and this is the one that is easy to miss - the auxiliary regression has to keep
the dummies as well. If it does not, the two covariance matrices come from different models
and the trace in $Q_{LM}$ is meaningless. So we extend the function.

In [ ]:
def VARLMtest(y, p, con, tr, exog, h):

    t, K = y.shape
    Beta, CovBeta, tratioBeta, residuals_u, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

    u_lags = lagmatrix(residuals_u, h)

    # keep the exogenous variables in the auxiliary regression too
    if isinstance(exog, np.ndarray):
        x_aux = np.hstack([exog, u_lags])
    else:
        x_aux = u_lags

    Beta, CovBeta, tratioBeta, residuals_e, X, SIGMA_e = VARlsExog(y, p, con, tr, x_aux)

    T = t - p
    LML = # FILL IN
    LMLpval = 1 - chi2.cdf(LML, h*K**2)

    m = # FILL IN
    q = # FILL IN
    s = # FILL IN
    N = # FILL IN

    FLMh = # FILL IN
    FLMh_pval = # FILL IN

    Results = [[LML, FLMh], [LMLpval, FLMh_pval], [h, h]]

    lm_table = pd.DataFrame({
        'Measure': pd.Categorical(['Test statistic', 'p-value', 'Lag order']),
        'Breusch_Godfrey': [row[0] for row in Results],
        'Edgerton_Shukur': [row[1] for row in Results]
    })

    return Results, lm_table, residuals_u


p = 3
con = 1
tr = 1
h = 4        # quarterly data
lag = 2

exog = dummies_full[p:, :]

Results, lm_table, residuals = VARLMtest(y, p, con, tr, exog, h)

lm_table

You should get:

| Measure         | Breusch-Godfrey | Edgerton-Shukur |
|-----------------|-----------------|-----------------|
| Test statistic  | 132.11          | 1.136           |
| p-value         | 0.0174          | 0.2185          |
| Lag order       | 4               | 4               |

**This is the interesting result of the class.**

Breusch-Godfrey rejects at 5%. Edgerton-Shukur does not, and not even close. Who is right?

Count the parameters. We have $T = 85$ usable observations and each equation has
$Kp + 2 + 3 = 20$ regressors. The auxiliary regression adds another $hK = 20$. The
asymptotic $\chi^2$ approximation behind $Q_{LM}$ assumes we have plenty of observations
per parameter, and here we do not. That is precisely the case Edgerton and Shukur (1999)
simulate, and they find the plain LM test rejects far too often. So we go with
Edgerton-Shukur.

Let us look at a few values of $h$.

In [ ]:
rows = []
for hh in [1, 2, 3, 4]:
    R, _, _ = VARLMtest(y, p, con, tr, exog, hh)
    rows.append([hh, R[0][0], R[1][0], R[0][1], R[1][1]])

pd.DataFrame(rows, columns=['h', 'BG stat', 'BG p-value', 'ES stat', 'ES p-value']).round(4)

|   h |   BG stat |   BG p-value |   ES stat |   ES p-value |
|-----|-----------|--------------|-----------|--------------|
|   1 |   18.9656 |       0.7988 |    0.5941 |       0.9389 |
|   2 |   65.5118 |       0.0695 |    1.0948 |       0.3202 |
|   3 |  102.9236 |       0.0179 |    1.2116 |       0.1403 |
|   4 |  132.1075 |       0.0174 |    1.1356 |       0.2185 |

Edgerton-Shukur never rejects, at any $h$. Breusch-Godfrey rejects for $h \geq 3$. That
pattern is what over-rejection looks like, rather than genuine seasonal autocorrelation.

Does the lag order from Exercise 2 survive? Try a few.

In [ ]:
rows = []
for pp in [1, 2, 3, 4, 5]:
    R, _, _ = VARLMtest(y, pp, con, tr, dummies_full[pp:, :], 4)
    rows.append([pp, R[0][0], R[1][0], R[0][1], R[1][1]])

pd.DataFrame(rows, columns=['p', 'BG stat', 'BG p-value', 'ES stat', 'ES p-value']).round(4)

|   p |   BG stat |   BG p-value |   ES stat |   ES p-value |
|-----|-----------|--------------|-----------|--------------|
|   1 |  137.2412 |       0.0080 |    1.4644 |       0.0079 |
|   2 |  143.4810 |       0.0029 |    1.4084 |       0.0167 |
| **3** |  132.1075 |       0.0174 |    1.1356 |   **0.2185** |
|   4 |  168.0524 |       0.0000 |    1.5295 |       0.0059 |
|   5 |  183.5960 |       0.0000 |    1.5037 |       0.0098 |

$p = 3$ is the only lag order in the range that Edgerton-Shukur does not reject at 5%.

So the misspecification test independently confirms the lag order we picked with the
information criteria in Exercise 2 - and the same order that Bjørnland (2009) uses.
Three different routes to the same answer.

Multivariate ARCH on the selected model.

In [ ]:
test, march_table = march(residuals, lag, K)

march_table

Test statistic 571.07 on 450 degrees of freedom, p-value 0.0001. Strong rejection.

Note the degrees of freedom: $qK^2(K+1)^2/4 = 2 \cdot 25 \cdot 36/4 = 450$, against only
85 observations. The test is being asked to estimate an enormous auxiliary system, so
treat the size of the statistic with some caution - but ARCH in quarterly macro data is
entirely normal and not something we can regress away.

Multivariate normality on the selected model.

In [ ]:
norm, multnorm_table = multnorm(residuals)

multnorm_table

| Test | Doornik_Hansen | Lutkepohl |
|---|---|---|
| Joint test statistic | 104.639 | 94.697 |
| P-value | 0.000 | 0.000 |
| Degrees of freedom | 10 | 10 |
| Skewness only | 15.714 | 8.607 |
| P-value | 0.008 | 0.126 |
| Kurtosis only | 88.925 | 86.090 |
| P-value | 0.000 | 0.000 |

Normality is rejected, and the breakdown shows why: kurtosis, not skewness (88.9 of the
104.6). The residuals have fat tails - a few large shocks. The 1992-93 currency crisis is
in the sample, which is exactly what the du92Q3 and du93Q1 dummies are there for, and
they evidently do not absorb all of it.

Note that the two versions disagree on skewness: Doornik-Hansen rejects at 1%, Lütkepohl
does not reject at all. The Cholesky factor used by Lütkepohl depends on the ordering of
the variables, which is a good reason to report Doornik-Hansen.

And the univariate cross-checks, which is what homework question 5 asks you to compare against.

In [ ]:
names = ['rfor', 'lgdp', 'pi', 'dlrexc', 'rdom']

rows = []
for i in range(K):
    lb = acorr_ljungbox(residuals[:, i], lags=[4], return_df=True)['lb_pvalue'].values[0]
    stat, pValue, fval, _ = het_arch(residuals[:, i], nlags=lag)
    (JB, JBpv, skew, kurtosis) = jarque_bera(residuals[:, i])
    rows.append([lb, stat, pValue, JB, JBpv])

univariate = pd.DataFrame(rows, index=names,
    columns=['Ljung-Box(4) p', 'ARCH(2) stat', 'ARCH(2) p', 'Jarque-Bera', 'JB p'])

univariate.round(4)

|        |   Ljung-Box(4) p |   ARCH(2) stat |   ARCH(2) p |   Jarque-Bera |   JB p |
|--------|------------------|----------------|-------------|---------------|--------|
| rfor   |           0.9212 |         3.2832 |      0.1937 |       17.6732 | 0.0001 |
| lgdp   |           0.9808 |         4.6743 |      0.0966 |       51.7238 | 0.0000 |
| pi     |           0.4600 |         1.9459 |      0.3780 |       20.6089 | 0.0000 |
| dlrexc |           0.4065 |         1.9109 |      0.3846 |        0.9887 | 0.6100 |
| rdom   |           0.3080 |         6.4790 |      0.0392 |       10.0415 | 0.0066 |

Reading this against the multivariate results:

* Autocorrelation: no single equation rejects Ljung-Box, consistent with Edgerton-Shukur
  and against Breusch-Godfrey. Another reason to trust the corrected test.
* ARCH: only rdom rejects at 5%, yet the multivariate test rejects hard. That is
  informative - the ARCH is in the *cross-equation* second moments, which the univariate
  tests cannot see. This is the best single argument for why we bother with multivariate
  tests at all.
* Normality: four of five equations reject, so the multivariate rejection is not driven
  by one outlier series.

Finally, stability of the selected model, using your own `comp()` and `stabVAR()`.

In [ ]:
Beta, CovBeta, tratioBeta, residuals, X, SIGMA_u = VARlsExog(y, p, con, tr, exog)

A = comp(Beta, p)
e = stabVAR(A)

print(f"A is {A.shape[0]}x{A.shape[1]}")
print(f"The five largest eigenvalue moduli: {np.round(e[:5], 4)}")
print(f"\nThe largest eigenvalue modulus is: {e[0]:.4f}")
if e[0] < 1:
    print("The VAR is stable")
else:
    print("The VAR is not stable")

$A$ is $15 \times 15$ and the five largest moduli are
[0.9465, 0.9318, 0.9318, 0.788, 0.788].

Stable, but the largest root is 0.9465 - close enough to the unit circle that it is worth
remembering when we get to cointegration in Exercise 4. A near-unit root in the companion
matrix is exactly what a cointegrated system looks like from the VAR side.

### Summary table for homework question 5

Selected model: VAR(3) in $[rfor,\, lgdp,\, \pi,\, dlrexc,\, rdom]$, constant, linear
trend and three dummies. $T = 85$ effective observations.

| Test | Statistic | df | p-value | Conclusion |
|---|---|---|---|---|
| Breusch-Godfrey LM, h=4 | 132.11 | 100 | 0.017 | rejects, but unreliable at T=85 |
| Edgerton-Shukur F, h=4 | 1.136 | (100, 229.1) | 0.219 | no autocorrelation |
| Doornik-Hendry MARCH, q=2 | 571.07 | 450 | 0.000 | ARCH present |
| Doornik-Hansen normality | 104.64 | 10 | 0.000 | rejected (kurtosis) |
| Lütkepohl normality | 94.70 | 10 | 0.000 | rejected (kurtosis) |
| Largest companion eigenvalue | 0.9465 | - | - | stable |

Points worth making in the write-up:

* The two autocorrelation tests disagree, and the small-sample corrected one is the one
  to trust here. Say why, do not just pick one.
* Rejecting normality does not invalidate the point estimates. OLS is still consistent.
  What it costs us is the exact finite-sample distribution of the test statistics, so the
  standard errors and confidence bands are asymptotic approximations only. For the
  bootstrap IRF confidence intervals later in the course it matters more.
* ARCH means the residuals are conditionally heteroskedastic. Again the point estimates
  survive; the efficiency claim and the usual covariance matrix do not.
* Autocorrelation would be the serious one, because it means the lag structure itself is
  wrong and the coefficients are inconsistent. That is the one we did not find.